# WP5: Climate Comparison

Extract ERA5 2 m air temperature and wind speed over the AOI and correlate with the ice area time series.

In [ ]:
import ee
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.insert(0, '..')
from src.utils import load_aoi, get_gee_project

ee.Initialize(project=get_gee_project())
aoi = load_aoi()

## 5.1 Extract ERA5 time series

In [ ]:
START, END = '2019-01-01', '2024-12-31'

era5 = (
    ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR')
    .filterBounds(aoi)
    .filterDate(START, END)
    .select(['temperature_2m', 'u_component_of_wind_10m', 'v_component_of_wind_10m'])
)

def extract_era5(image):
    stats = image.reduceRegion(reducer=ee.Reducer.mean(), geometry=aoi, scale=9000, maxPixels=1e9)
    return ee.Feature(None, stats.set('date', image.date().format('YYYY-MM-dd')))

era5_fc = era5.map(extract_era5)
era5_rows = era5_fc.getInfo()['features']
era5_df = pd.DataFrame([r['properties'] for r in era5_rows])
era5_df['date'] = pd.to_datetime(era5_df['date'])
era5_df = era5_df.sort_values('date').reset_index(drop=True)

# Convert temperature from K to °C
era5_df['t2m_c'] = era5_df['temperature_2m'] - 273.15
# Wind speed magnitude
era5_df['wind_speed'] = np.sqrt(era5_df['u_component_of_wind_10m']**2 + era5_df['v_component_of_wind_10m']**2)

era5_df.to_csv('../outputs/csv/era5_timeseries.csv', index=False)
era5_df.head()

## 5.2 Merge with ice area time series

In [ ]:
# ice_df = pd.read_csv('../outputs/csv/ice_area_timeseries.csv', parse_dates=['date'])
# merged = pd.merge_asof(ice_df.sort_values('date'), era5_df.sort_values('date'),
#                        on='date', direction='nearest', tolerance=pd.Timedelta('3d'))
# merged.head()

## 5.3 Pearson correlation & CCF

In [ ]:
# from scipy.stats import pearsonr
# r, p = pearsonr(merged['ice_area_km2'].dropna(), merged['t2m_c'].dropna())
# print(f'Pearson r (ice vs T2m): {r:.3f}, p={p:.3e}')

# # Cross-correlation
# ccf = [merged['ice_area_km2'].corr(merged['t2m_c'].shift(lag)) for lag in range(-30, 31)]
# plt.plot(range(-30, 31), ccf)
# plt.axvline(0, color='grey', lw=0.8)
# plt.xlabel('Lag (days)'); plt.ylabel('Pearson r'); plt.title('CCF: ice area vs T2m')
# plt.savefig('../outputs/figures/ccf_ice_t2m.png', dpi=150, bbox_inches='tight')
# plt.show()